In [ ]:
# ==============================================================================
# CELDA 0: RUTAS DEL REPOSITORIO
# Este cuaderno leia sus datos desde Google Drive. Ahora los lee del propio
# repositorio, de modo que corre en cualquier clon sin configuracion previa.
# La raiz se resuelve buscando hacia arriba: funciona igual si el cuaderno se
# ejecuta desde su carpeta o desde la raiz del proyecto.
# ==============================================================================
from pathlib import Path

def _raiz_del_repositorio() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'pyproject.toml').exists() and (carpeta / 'quanta').is_dir():
            return carpeta
    raise RuntimeError(
        'No se encontro la raiz del repositorio. Ejecute el cuaderno desde '
        'dentro del proyecto indice-sned.'
    )

RAIZ            = _raiz_del_repositorio()
RUTA_RAW        = (RAIZ / 'data' / 'raw').as_posix() + '/'
RUTA_PROCESADOS = (RAIZ / 'data' / 'processed').as_posix() + '/'
RUTA_REGISTRO   = (RAIZ / 'models' / 'registry').as_posix() + '/'
RUTA_METADATOS  = (RAIZ / 'models' / 'metadata').as_posix() + '/'

print('Raiz del repositorio:', RAIZ)


In [1]:
# ==============================================================================
# CELDA 1: LIBRERÍAS + INGESTA MATRÍCULA 2018-2019
# ==============================================================================
import pandas as pd

RUTA = RUTA_RAW + 'matricula/'

archivos = {
    2018: 'Resumen_matricula_UE_2018.csv',
    2019: 'Resumen_matricula_UE_2019.csv',
}

mat_brutas = {}
for anio, nombre in archivos.items():
    ruta = RUTA + nombre
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            mat_brutas[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"OK | {anio} | {mat_brutas[anio].shape[0]} filas x {mat_brutas[anio].shape[1]} columnas")

print("\nColumnas 2018:", mat_brutas[2018].columns.tolist())

Mounted at /content/drive
OK | 2018 | 26747 filas x 87 columnas
OK | 2019 | 27020 filas x 93 columnas

Columnas 2018: ['AGNO', 'RBD', 'DGV_RBD', 'NOM_RBD', 'COD_REG_RBD', 'COD_PRO_RBD', 'COD_COM_RBD', 'NOM_COM_RBD', 'COD_DEPROV_RBD', 'NOM_DEPROV_RBD', 'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'ESTADO_ESTAB', 'COD_ENSE', 'COD_ENSE2', 'MAT_HOM_1', 'MAT_MUJ_1', 'MAT_GRA_1', 'MAT_HOM_2', 'MAT_MUJ_2', 'MAT_GRA_2', 'MAT_HOM_3', 'MAT_MUJ_3', 'MAT_GRA_3', 'MAT_HOM_4', 'MAT_MUJ_4', 'MAT_GRA_4', 'MAT_HOM_5', 'MAT_MUJ_5', 'MAT_SI_5', 'MAT_GRA_5', 'MAT_HOM_6', 'MAT_MUJ_6', 'MAT_GRA_6', 'MAT_HOM_7', 'MAT_MUJ_7', 'MAT_GRA_7', 'MAT_HOM_8', 'MAT_MUJ_8', 'MAT_SI_8', 'MAT_GRA_8', 'MAT_HOM_TOT', 'MAT_MUJ_TOT', 'MAT_SI_TOT', 'MAT_TOTAL', 'CUR_SIM_01', 'CUR_SIM_02', 'CUR_SIM_03', 'CUR_SIM_04', 'CUR_SIM_05', 'CUR_SIM_06', 'CUR_SIM_07', 'CUR_SIM_08', 'CUR_SIM_TOT', 'CUR_COMB', 'MAT_JOR_MA', 'MAT_JOR_TA', 'MAT_JOR_MT', 'MAT_JOR_VE', 'MAT_JOR_NE', 'MAT_JOR_TOT', 'SIM_JOR_MA', 'SIM_JOR_TA', 'SIM_JOR_MT', 'SIM_JOR_V

In [3]:
# ==============================================================================
# CELDA 2 CORREGIDA: MATRÍCULA TOTAL Y TAMAÑO POR RBD (bienio 2018-19)
# ==============================================================================
def procesar_matricula(df):
    d = df.copy()
    d['RBD'] = pd.to_numeric(d['RBD'], errors='coerce')
    d = d.dropna(subset=['RBD'])
    d['rbd'] = d['RBD'].astype('Int64').astype(str)

    # Forzar conversión numérica (vienen como texto, posible separador de miles)
    d['matricula_total'] = pd.to_numeric(
        d['MAT_TOTAL'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )
    d['cursos_total'] = pd.to_numeric(d['CUR_SIM_TOT'], errors='coerce')

    return d[['rbd', 'matricula_total', 'cursos_total']]

mat_limpios = {anio: procesar_matricula(df) for anio, df in mat_brutas.items()}

for anio in mat_limpios:
    mat_limpios[anio] = mat_limpios[anio].groupby('rbd', as_index=False).mean()
    print(f"{anio}: {mat_limpios[anio].shape[0]} colegios únicos")

pool = pd.concat(mat_limpios.values(), ignore_index=True)
matricula_1819 = pool.groupby('rbd', as_index=False).mean()

print(f"\nBienio 2018-19: {matricula_1819.shape[0]} colegios")
print(matricula_1819.describe())

RUTA_SALIDA = RUTA_PROCESADOS
matricula_1819.to_parquet(RUTA_SALIDA + 'matricula_2018_19_por_rbd.parquet', index=False)
print("Guardado OK")

2018: 16044 colegios únicos
2019: 16236 colegios únicos

Bienio 2018-19: 16237 colegios
       matricula_total  cursos_total
count     16237.000000  10366.000000
mean         98.249957      5.744310
std         139.463870      4.013387
min           0.000000      1.000000
25%           0.000000      3.000000
50%          49.500000      5.000000
75%         139.500000      7.666667
max        2153.750000     50.750000
Guardado OK


In [4]:
# ==============================================================================
# CELDA 3: INTEGRAR MATRÍCULA A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = RUTA_PROCESADOS
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v7.parquet')
mat = pd.read_parquet(RUTA + 'matricula_2018_19_por_rbd.parquet')

df_modelo_v8 = pd.merge(df_modelo, mat, on='rbd', how='left', validate='one_to_one')

print(f"Filas: {len(df_modelo_v8)} (antes: {len(df_modelo)})")
print(f"Con dato de matrícula: {df_modelo_v8['matricula_total'].notna().sum()}")

df_modelo_v8.to_parquet(RUTA + 'tabla_modelo_final_v8.parquet', index=False)
print(df_modelo_v8.shape)

Filas: 7754 (antes: 7754)
Con dato de matrícula: 7754
(7754, 57)
